In [1]:
import os
from datetime import UTC, datetime, timedelta

import cabaret
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits


In [2]:
## Observatory components

site = cabaret.Site(
    sky_background=150,  # e-/m^2/arcsec^2/s
    seeing=1.35,  # arcseconds
    elevation=2500,  # meters
)

telescope = cabaret.Telescope(
    focal_length=8,  # meters
    diameter=1,  # meters
)

pixel_defects = None
camera = cabaret.Camera(
    name="Simulated SPECULOOS Camera",
    width=2048,  # pixels
    height=2048,  # pixels
    bin_x=1,  # pixels
    bin_y=1,  # pixels
    pitch=13.5,  # microns
    max_adu=2**16 - 1,  # ADU
    well_depth=2**16 - 1,  # electrons
    bias=300,  # ADU
    gain=1,  # electrons per ADU
    read_noise=6.2,  # electrons
    dark_current=0.2,  # electrons per second
    average_quantum_efficiency=1,  # fraction
    rotation=0,  # degrees
    pixel_defects=pixel_defects,
)

In [3]:
observatory = cabaret.Observatory(
    name="Simulated SPECULOOS",
    site=site,
    telescope=telescope,
    camera=camera,
)

camera.set_plate_scale_from_focal_length(telescope.focal_length)

In [4]:
if_pds = 1 if pixel_defects is not None else 0
print(f"Pixel defects: {'enabled' if if_pds else 'disabled'}")

Pixel defects: disabled


In [5]:
# find coordinates of Gaia DR3 5461814875584141056
from astroquery.mast import Mast

query_string = "Gaia DR3 5461814875584141056"
mast = Mast()
target_radec = mast.resolve_object(query_string)
target_radec

<SkyCoord (ICRS): (ra, dec) in deg
    (153.02702278, -30.82398023)>

In [6]:
depth = 0.01  # 99% transit depth
duration = 1  # 1 hour transit duration

exp_time = 39  # seconds
readout_time = 4.5  # seconds

time = np.arange(
    -duration * 1.5,
    duration * 1.5,
    (exp_time + readout_time) / 3600,
)

transit = np.ones_like(time)
transit[(time > -duration / 2) & (time < duration / 2)] -= depth


In [7]:
transit

array([1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
       1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
       1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
       1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
       1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
       1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
       1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
       1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 0.99, 0.99, 0.99, 0.99, 0.99,
       0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99,
       0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99,
       0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99,
       0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99,
       0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99,
       0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.

In [8]:
seed = -1
n_imgs = len(transit)
n_imgs_cal = 10

ra = target_radec.ra.deg  # degrees
dec = target_radec.dec.deg  # degrees
image_run = f"{ra}_{dec}_s{seed}_n{n_imgs}_nc{n_imgs_cal}_{exp_time}s_r{readout_time}s_pds{if_pds}"

filter_band = cabaret.Filters.RP
dateobs_0 = datetime(2024, 6, 1, 0, 0, 0, tzinfo=UTC)

center = SkyCoord(ra=ra, dec=dec, unit="deg")

sources = cabaret.GaiaQuery.get_sources(
    center=center,
    radius=camera.get_fov_radius() * 1.5,
    filter_band=filter_band,
    limit=10000,
    dateobs=dateobs_0,
    # timeout=60,
)


In [9]:
# find source with Gaia DR3 5461814875584141056 via argmin of angular separation

target_source_ind = np.argmin(sources.coords.separation(center))
print(f"Target source: {sources[target_source_ind]}")

Target source: Sources(coords=<SkyCoord (ICRS): (ra, dec) in deg
    (153.02595228, -30.82373566)>, fluxes=array([34405.61458412]))


In [10]:
target_source_ind

97

In [ ]:
# Light frames
original_flux = sources.fluxes[target_source_ind]
os.makedirs(f"data/{image_run}", exist_ok=True)
# Generate the FITS image for this filter
for j in range(n_imgs):
    print(f"Generating light frame {j + 1}/{n_imgs} for filter {filter_band.name}...")
    dateobs = dateobs_0 + timedelta(seconds=j * (exp_time + readout_time))

    sources.fluxes[target_source_ind] = original_flux * transit[j]

    ## Light frame
    hdu = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=exp_time,  # seconds
        dateobs=dateobs,
        sources=sources,
        fwhm_multiplier=7,  # to determine the rendering radius around each star
        user_header={
            "FILTER": filter_band.name,
            "IMAGETYP": "Light Frame",
            "object": image_run,
        },
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu.writeto(
        f"data/{image_run}/{image_run}_{filter_band.name}_Light_{j}.fits",
        overwrite=True,
    )

# Calibration frames (bias, dark, flat)
os.makedirs(f"data/{image_run}", exist_ok=True)
for j in range(n_imgs_cal):
    print(f"Generating calibration frames {j + 1}/{n_imgs_cal}")
    dateobs = dateobs_0 + timedelta(seconds=j * (exp_time + readout_time))

    ## Bias frame
    hdu_bias = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=0,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "Bias Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_bias[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_bias.writeto(
        f"data/{image_run}/Bias_{j}.fits",
        overwrite=True,
    )

    ## Dark frame
    hdu_dark = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=exp_time,  # seconds
        dateobs=dateobs,
        light=0,
        sources=None,
        user_header={"IMAGETYP": "Dark Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_dark[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_dark.writeto(
        f"data/{image_run}/Dark_{j}.fits",
        overwrite=True,
    )

    ## Flat frame
    site.sun_altitude = -4
    hdu_flat = observatory.generate_fits_image(
        ra=ra,  # degrees
        dec=dec,  # degrees
        exp_time=0.5,  # seconds
        dateobs=dateobs,
        light=1,
        sources=cabaret.Sources.from_arrays(ra=[], dec=[], fluxes=[]),
        user_header={"FILTER": filter_band.name, "IMAGETYP": "Flat Frame"},
        seed=seed if seed >= 0 else None,
    )

    # add comment about the pixel defects
    hdu_flat[0].header.add_comment(f"Pixel defects: {pixel_defects}")

    hdu_flat.writeto(
        f"data/{image_run}/Flat_{filter_band.name}_{j}.fits",
        overwrite=True,
    )
    site.sun_altitude = None

In [13]:
image_run

'153.02702277876_-30.82398022844_s-1_n249_nc10_39s_r4.5s_pds0'

In [12]:
print(
    f'uv run pipeline.py --image_path ./data/{image_run} --target {image_run} --query-string "Gaia DR3 5461814875584141056" --report --output-dir results/'
)


uv run pipeline.py --image_path ./data/153.02702277876_-30.82398022844_s-1_n249_nc10_39s_r4.5s_pds0 --target 153.02702277876_-30.82398022844_s-1_n249_nc10_39s_r4.5s_pds0 --query-string "Gaia DR3 5461814875584141056" --report --output-dir results/
